In [1]:
import pandas as pd

dftransactions = pd.read_csv("LI-Small_Trans.csv")

dfaccounts = pd.read_csv("LI-Small_accounts.csv")

print(dftransactions.head())
#print(dfaccounts.head())


          Timestamp  From Bank    Account  To Bank  Account.1  \
0  2022/09/01 00:08         11  8000ECA90       11  8000ECA90   
1  2022/09/01 00:21       3402  80021DAD0     3402  80021DAD0   
2  2022/09/01 00:00         11  8000ECA90     1120  8006AA910   
3  2022/09/01 00:16       3814  8006AD080     3814  8006AD080   
4  2022/09/01 00:00         20  8006AD530       20  8006AD530   

   Amount Received Receiving Currency  Amount Paid Payment Currency  \
0       3195403.00          US Dollar   3195403.00        US Dollar   
1          1858.96          US Dollar      1858.96        US Dollar   
2        592571.00          US Dollar    592571.00        US Dollar   
3            12.32          US Dollar        12.32        US Dollar   
4          2941.56          US Dollar      2941.56        US Dollar   

  Payment Format  Is Laundering  
0   Reinvestment              0  
1   Reinvestment              0  
2         Cheque              0  
3   Reinvestment              0  
4   Reinvest

I - Profiling

In [2]:
dfaccounts.dtypes

Bank Name           str
Bank ID           int64
Account Number      str
Entity ID           str
Entity Name         str
dtype: object

In [3]:
dftransactions.dtypes

Timestamp                 str
From Bank               int64
Account                   str
To Bank                 int64
Account.1                 str
Amount Received       float64
Receiving Currency        str
Amount Paid           float64
Payment Currency          str
Payment Format            str
Is Laundering           int64
dtype: object

In [4]:
dftransactions = dftransactions.drop(columns = "Is Laundering")

Now I am going to count distinct and unique values for each column of the transactions table.
Here we have to be careful because "unique" in Pandas equals "distinct" in PowerBI.

In [5]:
# Distinct values

print(dftransactions.nunique())




Timestamp               14533
From Bank               41814
Account                681281
To Bank                 21588
Account.1              576176
Amount Received       1194921
Receiving Currency         15
Amount Paid           1204309
Payment Currency           15
Payment Format              7
dtype: int64


So for obtaining unique values (i.e. values in a column that appear just once), I should get first the value counts and then count how many values are "repeated" just one time.

In [6]:
ValueCounts= {}

for i in dftransactions:
    ValueCounts[i] = dftransactions[i].value_counts()
  


In [7]:
print(ValueCounts)

{'Timestamp': Timestamp
2022/09/01 00:22    15221
2022/09/01 00:20    15070
2022/09/01 00:01    15062
2022/09/01 00:21    15061
2022/09/01 00:14    15049
                    ...  
2022/09/11 12:39        1
2022/09/13 01:45        1
2022/09/13 13:15        1
2022/09/11 09:57        1
2022/09/14 10:53        1
Name: count, Length: 14533, dtype: int64, 'From Bank': From Bank
70        609991
11        123456
20        120114
14         64681
12         64465
           ...  
371503         1
374271         1
374508         1
373565         1
376947         1
Name: count, Length: 41814, dtype: int64, 'Account': Account
10042B660    222037
10042B6A8    138777
10042B6F0     42385
10042B780     30802
10042BA51     28932
              ...  
80D2860B0         1
80D284FA0         1
80D27FCA0         1
81B172D21         1
81B3D58E1         1
Name: count, Length: 681281, dtype: int64, 'To Bank': To Bank
11        66055
20        56805
14        36313
12        34960
18        34178
          ...  

In [8]:
print((ValueCounts["Timestamp"] == 1))

Timestamp
2022/09/01 00:22    False
2022/09/01 00:20    False
2022/09/01 00:01    False
2022/09/01 00:21    False
2022/09/01 00:14    False
                    ...  
2022/09/11 12:39     True
2022/09/13 01:45     True
2022/09/13 13:15     True
2022/09/11 09:57     True
2022/09/14 10:53     True
Name: count, Length: 14533, dtype: bool


In [9]:
uniqueValues = {}
for i in ValueCounts: #every i is a key from a dictionary and its value is a dataframe that can be manipulated with Pandas
    uniqueValues[i] = ValueCounts[i][ValueCounts[i] == 1].count()


In [10]:
print(uniqueValues)

{'Timestamp': np.int64(53), 'From Bank': np.int64(4714), 'Account': np.int64(211825), 'To Bank': np.int64(5156), 'Account.1': np.int64(157062), 'Amount Received': np.int64(615389), 'Receiving Currency': np.int64(0), 'Amount Paid': np.int64(621441), 'Payment Currency': np.int64(0), 'Payment Format': np.int64(0)}


Checking if everytime a bank name is repeated, the same ID is assigned to that bank.

In [11]:
bankNamesGroupinbAndCountingValues = dfaccounts.groupby("Bank Name")["Bank ID"].nunique()
bankNamesGroupinbAndCountingValues

Bank Name
Acme Bancorp             22
Acme Bank                15
Acme Community Bank      19
Acme Cooperative Bank    15
Acme Credit Union        14
                         ..
Willows Credit Union     15
Willows Federal Bank     28
Willows Savings Bank     15
Willows Thrift           17
Willows Trust Bank        8
Name: Bank ID, Length: 27652, dtype: int64

In [12]:
bankNamesGroupinbAndCountingValues[bankNamesGroupinbAndCountingValues != 1].count()

np.int64(468)

Looking closer at Arbor Bank different IDs.

In [13]:
x = dfaccounts[dfaccounts["Bank Name"] == "Arbor Bank"]["Bank ID"].value_counts()
x

Bank ID
122779    270
3612       23
312401     10
320799      7
329121      7
316160      5
359004      4
347680      2
363221      2
370290      1
363400      1
361130      1
Name: count, dtype: int64

Now I want to count values in the Bank Name column from the Accounts table to check if the relation Bank ID → Bank Name is a function.

In [14]:
bankIDGrouping = dfaccounts.groupby("Bank ID")["Bank Name"].nunique()
bankIDGrouping[bankIDGrouping != 1].count

<bound method Series.count of Series([], Name: Bank Name, dtype: int64)>

Checking which rows are related to more than one account number.

In [15]:
a = dfaccounts["Account Number"].value_counts()

b = a[a != 1]

b

Account Number
817037B20    2
817038DF0    2
8177C8ED0    2
8177C94B0    2
Name: count, dtype: int64

Selecting all the rows of the dataframe which have repeated account numbers.

In [16]:
c = b.index
repeatedAccountNumbers = dfaccounts[dfaccounts["Account Number"].isin(c)]

repeatedAccountNumbers

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
364085,Savings Bank of the South,61144,817037B20,8007D0D30,Corporation #54169
434859,Savings Bank of the South,61144,817038DF0,80100DA30,Partnership #788
547803,Spain Bank #507,144720,8177C8ED0,800C73BF0,Corporation #38102
555545,Spain Bank #507,144720,8177C94B0,800981A00,Partnership #32750
569902,Savings Bank of Columbus,113213,817037B20,80129EC80,Corporation #795
571683,Estonia Bank #2218,144840,8177C8ED0,800B01A00,Sole Proprietorship #31292
573683,Estonia Bank #2218,144840,8177C94B0,800BB3280,Corporation #35681
585843,Savings Bank of Columbus,113213,817038DF0,8014110E0,Corporation #11573


Checking duplicated accounts.

In [17]:
repeatedAccountNumbers[["Bank ID","Account Number"]].duplicated()

364085    False
434859    False
547803    False
555545    False
569902    False
571683    False
573683    False
585843    False
dtype: bool

Checking Entity ID and Entity Name: 224,931 distinct and 145,348 unique values.
These two columns share the same amount of distinct and unique values.
I.e.: is the relation between these columns a bijective function?
To check this I would have to see with how many names each ID is related.

In [18]:
dfaccounts["Entity ID"]

0         800D8CCF0
1         800B505E0
2         800D03F60
3         801567C10
4         801085E00
            ...    
712683    800D080A0
712684    8005319C0
712685    800453480
712686    8003B5120
712687    8003122E0
Name: Entity ID, Length: 712688, dtype: str

Checking:

1) if Payment Currency ≠ Receiving Currency then Amount Paid ≠ Amount Received

2) if Payment Currency = Receiving Currency then Amount Paid = Amount Received


In [19]:
sameCurrency = dftransactions[dftransactions["Payment Currency"] == dftransactions["Receiving Currency"]][["Amount Paid","Amount Received"]]

#sameCurrency

#sameCurrency has 6,825,173 / 6,924,049 rows

sameCurrency[sameCurrency["Amount Paid"] == sameCurrency["Amount Received"]] #it has 6,825,173 rows, so in all of them both attributes are equal.

,Amount Paid,Amount Received
0,3.195403e+06,3.195403e+06
1,1.858960e+03,1.858960e+03
2,5.925710e+05,5.925710e+05
3,1.232000e+01,1.232000e+01
4,2.941560e+03,2.941560e+03
...,...,...
6924044,3.346900e-02,3.346900e-02
6924045,1.313000e-03,1.313000e-03
6924046,1.305800e-02,1.305800e-02
6924047,4.145370e-01,4.145370e-01


In [20]:
difCurrency = dftransactions[dftransactions["Payment Currency"] != dftransactions["Receiving Currency"]][["Timestamp", "Payment Currency", "Receiving Currency","Amount Paid","Amount Received"]]
difCurrency
#difCurrency.shape

# difCurrency has 98,876 / 6,924,049 rows.
# 6,825,173 + 98,876 = 6,924,049 rows.

difCurrency[difCurrency["Amount Paid"] != difCurrency["Amount Received"]] #it has 98,858 rows, which means there are 18 rows with different currencies and equal Amount Paid, Amount Received.

,Timestamp,Payment Currency,Receiving Currency,Amount Paid,Amount Received
2770,2022/09/01 00:12,US Dollar,Euro,55.79,47.610000
8081,2022/09/01 00:28,US Dollar,Yuan,142.53,954.620000
10451,2022/09/01 00:18,US Dollar,Yen,160.63,16930.030000
12948,2022/09/01 00:17,US Dollar,UK Pound,18.76,14.520000
13799,2022/09/01 00:02,US Dollar,Euro,43.35,37.000000
...,...,...,...,...,...
6924007,2022/09/10 23:57,Yuan,Bitcoin,0.39,0.000005
6924009,2022/09/10 23:30,Yuan,Bitcoin,0.55,0.000007
6924019,2022/09/10 23:38,US Dollar,Bitcoin,0.08,0.000007
6924021,2022/09/10 23:31,US Dollar,Bitcoin,0.23,0.000020


Let's see those 18 rows with different currency and equal amounts

In [21]:

difCurrency[difCurrency["Amount Paid"] == difCurrency["Amount Received"]][["Timestamp", "Payment Currency", "Receiving Currency", "Amount Paid","Amount Received"]]

,Timestamp,Payment Currency,Receiving Currency,Amount Paid,Amount Received
89998,2022/09/01 00:15,UK Pound,US Dollar,0.03,0.03
137951,2022/09/01 00:08,Canadian Dollar,US Dollar,0.01,0.01
162050,2022/09/01 00:18,UK Pound,US Dollar,0.01,0.01
236263,2022/09/01 00:13,UK Pound,Euro,0.02,0.02
314776,2022/09/01 00:07,Rupee,Yen,0.01,0.01
469282,2022/09/01 00:51,Yuan,Saudi Riyal,0.01,0.01
653040,2022/09/01 04:32,Canadian Dollar,Saudi Riyal,0.01,0.01
728900,2022/09/01 06:44,Swiss Franc,US Dollar,0.01,0.01
841429,2022/09/01 09:14,Canadian Dollar,US Dollar,0.02,0.02
858188,2022/09/01 09:24,Brazil Real,Shekel,0.01,0.01


We can see that most of these transactions (#16) have been done during the same day (2022/09/01) while 2 of them have been done in different months (february and august). 
But we can also see that the amounts are really low, so the explanation to this observation is probably that all involve amounts equal or lower than 0.03, 
consistent with rounding at negligible values. Not material to the analysis.



In [22]:
print(dfaccounts.columns)
dfaccounts.head()

Index(['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name'], dtype='str')


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #2820,314693,81B86A280,800D8CCF0,Corporation #41344
1,France Bank #4585,311253,8187FEA80,800B505E0,Corporation #54497
2,China Bank #2242,39996,803961E00,800D03F60,Partnership #36904
3,National Bank of Newport,331440,81B075800,801567C10,Corporation #16224
4,UK Bank #33,135417,80CF87C80,801085E00,Partnership #72930


Primary key - Accounts table

Now I am going to check if the account table's primary key is different in every row.

In [23]:
AccountsPrimaryKey = dfaccounts.duplicated(subset = ["Bank ID","Account Number"])

AccountsPrimaryKey[AccountsPrimaryKey != False]

Series([], dtype: bool)

As we can see, there are no results when we look for True values, meaning there are no repeated rows regarding these two attributes.

Primary key - Transactions table

Since there is no natural primary key in dftransactions, I will create a synthetic one.

In [24]:
dftransactions = dftransactions.reset_index(names="TransactionID")

Checking the relation dfaccounts.Entity ID --> dfaccounts.Entity Names

In [25]:
EntitiesIDGrouping = dfaccounts.groupby("Entity ID")["Entity Name"].nunique()

EntitiesIDGrouping[EntitiesIDGrouping != 1]

Series([], Name: Entity Name, dtype: int64)

This shows that every ID is related with one Entity Name, which is what we suspected just by looking at the first distinct and unique values observations.

Cross-table referential integrity check

1) (dftransactions.From Bank, dftrasactions.Account) → (dfaccounts.Bank ID, dfaccounts.Account Number)

In [26]:
mergedTable1 = dftransactions.merge(dfaccounts, left_on = ["From Bank","Account"], right_on = ["Bank ID","Account Number"], how = "left")

mergedTable1[["Bank ID","Account Number"]].isna().any()


Bank ID           False
Account Number    False
dtype: bool

Cross-table referential integrity check

2) (dftransactions.To Bank, dftrasactions.Account1) → (dfaccounts.Bank ID, dfaccounts.Account Number)

In [27]:
mergedTable2 = dftransactions.merge(dfaccounts, left_on = ["To Bank","Account.1"], right_on = ["Bank ID","Account Number"], how = "left")

mergedTable2[["Bank ID","Account Number"]].isna().any()



Bank ID           False
Account Number    False
dtype: bool

II - Cleaning


Checking if there are empty rows in dftransactions

In [28]:
dftransactions.isna().any().any()

np.False_

Renaming columns with composite names separated by a space

In [29]:
dfaccounts = dfaccounts.rename(columns={
    "Bank Name": "bank_name",
    "Bank ID": "bank_id",
    "Account Number": "account_number",
    "Entity ID": "entity_id",
    "Entity Name": "entity_name"
})

dftransactions = dftransactions.rename(columns={
    "TransactionID": "id",
    "From Bank": "from_bank",
    "To Bank": "to_bank",
    "Account.1": "to_account",
    "Amount Received": "amount_received",
    "Receiving Currency": "receiving_currency",
    "Amount Paid": "amount_paid",
    "Payment Format": "payment_format",
    "Timestamp": "timestamp",
    "Account": "account",
    "Payment Currency": "payment_currency"
})


Converting data type in the dfaccounts table

In [30]:
dfaccounts["bank_id"] = dfaccounts["bank_id"].astype(str)

Converting data type in the dfaccounts table

In [31]:
dftransactions["from_bank"] = dftransactions["from_bank"].astype(str)
dftransactions["to_bank"] = dftransactions["to_bank"].astype(str)

In [32]:
dftransactions["timestamp"] = pd.to_datetime(dftransactions["timestamp"])

In [33]:
dftransactions["id"] = dftransactions["id"].astype(str)

Checking duplicates

In [34]:
print(dfaccounts.duplicated().any())
print(dftransactions.drop(columns="id").duplicated().any())

False
True


In [35]:
dftransactions[dftransactions.drop(columns="id").duplicated(keep = False)].count()

id                    16
timestamp             16
from_bank             16
account               16
to_bank               16
to_account            16
amount_received       16
receiving_currency    16
amount_paid           16
payment_currency      16
payment_format        16
dtype: int64

Cleaning Final Validation

In [36]:
dfaccounts.columns

Index(['bank_name', 'bank_id', 'account_number', 'entity_id', 'entity_name'], dtype='str')

In [37]:
dftransactions.columns

Index(['id', 'timestamp', 'from_bank', 'account', 'to_bank', 'to_account',
       'amount_received', 'receiving_currency', 'amount_paid',
       'payment_currency', 'payment_format'],
      dtype='str')

In [38]:
dfaccounts.dtypes


bank_name         str
bank_id           str
account_number    str
entity_id         str
entity_name       str
dtype: object

In [39]:
dftransactions.dtypes

id                               str
timestamp             datetime64[us]
from_bank                        str
account                          str
to_bank                          str
to_account                       str
amount_received              float64
receiving_currency               str
amount_paid                  float64
payment_currency                 str
payment_format                   str
dtype: object

In [40]:
dfaccounts.shape

(712688, 5)

In [41]:
dftransactions.shape

(6924049, 11)

FAN-IN

Checking how many different accounts each “to_account” account is receiving transactions from.


In [42]:
distinctSendersPerAccount = dftransactions.groupby("to_account")["account"].nunique()

distinctSendersPerAccount

# here I see 576,176 results. A lot of them are related just to one or two accounts. 




to_account
10042B660    785
10042B6A8    478
10042B6F0    176
10042B738     70
10042B780    118
            ... 
81C1EC5B0      1
81C1EC600      1
81C1EC650      2
81C1EC6A0      1
81C1EC6F1      2
Name: account, Length: 576176, dtype: int64

In [43]:
distinctSendersPerAccount.value_counts() #how many accounts receive transactions from #account distinct senders.

account
1      223040
3      138043
2      121290
4       57517
5       19396
        ...  
176         1
118         1
85          1
88          1
89          1
Name: count, Length: 93, dtype: int64

In [44]:
distinctSendersPerAccount[distinctSendersPerAccount > 2].value_counts()

# We are left with 231,846 results.

account
3      138043
4       57517
5       19396
6        7011
7        2901
        ...  
176         1
118         1
85          1
88          1
89          1
Name: count, Length: 91, dtype: int64

Filtering distinctSendersPerAccount in the original dataframe

In [45]:
x_1 = distinctSendersPerAccount[distinctSendersPerAccount > 2]

focusGroup = dftransactions[dftransactions["to_account"].isin(x_1.index)]

focusGroup

,id,timestamp,from_bank,account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format
0,0,2022-09-01 00:08:00,11,8000ECA90,11,8000ECA90,3.195403e+06,US Dollar,3.195403e+06,US Dollar,Reinvestment
4,4,2022-09-01 00:00:00,20,8006AD530,20,8006AD530,2.941560e+03,US Dollar,2.941560e+03,US Dollar,Reinvestment
5,5,2022-09-01 00:24:00,12,8006ADD30,12,8006ADD30,6.473620e+03,US Dollar,6.473620e+03,US Dollar,Reinvestment
6,6,2022-09-01 00:17:00,11,800059120,1217,8006AD4E0,6.056200e+04,US Dollar,6.056200e+04,US Dollar,ACH
7,7,2022-09-01 00:07:00,11,8000ECA90,11,8000ECA90,2.297000e+01,US Dollar,2.297000e+01,US Dollar,Reinvestment
...,...,...,...,...,...,...,...,...,...,...,...
6924043,6924043,2022-09-10 23:45:00,70997,81AD76C91,73607,81C00EA71,1.502880e-01,Bitcoin,1.502880e-01,Bitcoin,Bitcoin
6924045,6924045,2022-09-10 23:48:00,271241,81B567481,173457,81C0DA751,1.313000e-03,Bitcoin,1.313000e-03,Bitcoin,Bitcoin
6924046,6924046,2022-09-10 23:50:00,271241,81B567481,173457,81C0DA751,1.305800e-02,Bitcoin,1.305800e-02,Bitcoin,Bitcoin
6924047,6924047,2022-09-10 23:57:00,170558,81A2206B1,275798,81C1D5CA1,4.145370e-01,Bitcoin,4.145370e-01,Bitcoin,Bitcoin


Time Window

In [46]:
focusGroup["timestamp"].describe()



count                       5318124
mean     2022-09-05 11:53:29.441793
min             2022-09-01 00:00:00
25%             2022-09-02 10:19:00
50%             2022-09-05 18:28:00
75%             2022-09-08 05:32:00
max             2022-09-17 15:28:00
Name: timestamp, dtype: object

Approximately 16 days

Until 02: 25%
Until 05: 50%
Until 08: 75%
Until 17: 100%

They are more concentrated (75%) in the first half of the studied time window where they are evenly distributed.

Creating a new table in which the first column will show the receiving accounts and the second column will show the timestamp at which the transaction has been made and sorting its values.

In [47]:
time_table_1 = focusGroup[["to_account", "timestamp"]]
time_table_1

,to_account,timestamp
0,8000ECA90,2022-09-01 00:08:00
4,8006AD530,2022-09-01 00:00:00
5,8006ADD30,2022-09-01 00:24:00
6,8006AD4E0,2022-09-01 00:17:00
7,8000ECA90,2022-09-01 00:07:00
...,...,...
6924043,81C00EA71,2022-09-10 23:45:00
6924045,81C0DA751,2022-09-10 23:48:00
6924046,81C0DA751,2022-09-10 23:50:00
6924047,81C1D5CA1,2022-09-10 23:57:00


In [48]:
time_table_1 = time_table_1.sort_values(["to_account", "timestamp"])
time_table_1

,to_account,timestamp
157311,10042B660,2022-09-01 00:05:00
7112,10042B660,2022-09-01 00:08:00
72921,10042B660,2022-09-01 00:13:00
164374,10042B660,2022-09-01 00:13:00
151798,10042B660,2022-09-01 00:18:00
...,...,...
5389191,81C1EC560,2022-09-08 10:27:00
6038717,81C1EC560,2022-09-09 07:02:00
6038716,81C1EC560,2022-09-09 07:13:00
6619796,81C1EC560,2022-09-09 23:01:00


In [49]:
time_table_1["ts_min"] = time_table_1["timestamp"].astype("int64") / 60_000_000_000
time_table_1

,to_account,timestamp,ts_min
157311,10042B660,2022-09-01 00:05:00,27699.845
7112,10042B660,2022-09-01 00:08:00,27699.848
72921,10042B660,2022-09-01 00:13:00,27699.853
164374,10042B660,2022-09-01 00:13:00,27699.853
151798,10042B660,2022-09-01 00:18:00,27699.858
...,...,...,...
5389191,81C1EC560,2022-09-08 10:27:00,27710.547
6038717,81C1EC560,2022-09-09 07:02:00,27711.782
6038716,81C1EC560,2022-09-09 07:13:00,27711.793
6619796,81C1EC560,2022-09-09 23:01:00,27712.741


Gap

In [50]:
time_table_1["gap"] = time_table_1.groupby("to_account")["timestamp"].diff()
time_table_1

,to_account,timestamp,ts_min,gap
157311,10042B660,2022-09-01 00:05:00,27699.845,NaT
7112,10042B660,2022-09-01 00:08:00,27699.848,0 days 00:03:00
72921,10042B660,2022-09-01 00:13:00,27699.853,0 days 00:05:00
164374,10042B660,2022-09-01 00:13:00,27699.853,0 days 00:00:00
151798,10042B660,2022-09-01 00:18:00,27699.858,0 days 00:05:00
...,...,...,...,...
5389191,81C1EC560,2022-09-08 10:27:00,27710.547,0 days 00:09:00
6038717,81C1EC560,2022-09-09 07:02:00,27711.782,0 days 20:35:00
6038716,81C1EC560,2022-09-09 07:13:00,27711.793,0 days 00:11:00
6619796,81C1EC560,2022-09-09 23:01:00,27712.741,0 days 15:48:00


In [51]:
time_table_1["gap"] = time_table_1["gap"].dt.total_seconds()/60
time_table_1

,to_account,timestamp,ts_min,gap
157311,10042B660,2022-09-01 00:05:00,27699.845,NaN
7112,10042B660,2022-09-01 00:08:00,27699.848,3.0
72921,10042B660,2022-09-01 00:13:00,27699.853,5.0
164374,10042B660,2022-09-01 00:13:00,27699.853,0.0
151798,10042B660,2022-09-01 00:18:00,27699.858,5.0
...,...,...,...,...
5389191,81C1EC560,2022-09-08 10:27:00,27710.547,9.0
6038717,81C1EC560,2022-09-09 07:02:00,27711.782,1235.0
6038716,81C1EC560,2022-09-09 07:13:00,27711.793,11.0
6619796,81C1EC560,2022-09-09 23:01:00,27712.741,948.0


Timestamp Dispersion for Each Receiving Account


In [52]:
timestampDeviation = time_table_1.groupby("to_account")["ts_min"].std()
timestampDeviation

to_account
10042B660    5.081980
10042B6A8    5.078705
10042B6F0    5.113733
10042B738    5.122919
10042B780    5.216621
               ...   
81C1EAD61    4.536316
81C1EB101    4.449697
81C1EB521    4.321005
81C1EC470    1.704046
81C1EC560    4.250851
Name: ts_min, Length: 231846, dtype: float64

In [53]:
timestampDeviation.describe()

count    231846.000000
mean          4.441294
std           0.793064
min           0.000577
25%           4.275753
50%           4.460382
75%           4.671266
max          13.522487
Name: ts_min, dtype: float64

Looking at what is going on at lower percentiles

In [54]:
timestampDeviation.quantile([0.01, 0.02, 0.03, 0.04, 0.05, 0.10, 0.15, 0.20])

0.01    0.553701
0.02    1.350443
0.03    2.301883
0.04    3.121694
0.05    3.755335
0.10    4.103882
0.15    4.178762
0.20    4.231626
Name: ts_min, dtype: float64

Isolating percentile 1  

In [55]:
corte = timestampDeviation.quantile(0.01)
timestampDeviation_percentile1 = timestampDeviation[timestampDeviation <= corte]
timestampDeviation_percentile1.describe()




count    2319.000000
mean        0.210483
std         0.189415
min         0.000577
25%         0.009730
50%         0.186936
75%         0.385480
max         0.553392
Name: ts_min, dtype: float64

Building a Percentiles Comparison Table

In [56]:
percentiles = timestampDeviation.quantile([0.0025, 0.0020, 0.0030,0.0035, 0.0040, 0.0050, 0.0075, 0.01, 0.02, 0.03, 0.04, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75])
percentiles_comparison_table = percentiles.to_frame(name="std_min")
percentiles_comparison_table["ratio_to_previous"] = percentiles_comparison_table["std_min"] / percentiles_comparison_table["std_min"].shift(1)
percentiles_comparison_table

,std_min,ratio_to_previous
0.0025,0.009731,NaN
0.0020,0.008976,0.922412
0.0030,0.011012,1.226866
0.0035,0.032581,2.958558
0.0040,0.091119,2.796711
0.0050,0.186956,2.051785
0.0075,0.385542,2.062210
0.0100,0.553701,1.436162
0.0200,1.350443,2.438941
0.0300,2.301883,1.704539


In [57]:
percentileCut = timestampDeviation.quantile(0.004)
thresholdSet = timestampDeviation[timestampDeviation <= percentileCut]
thresholdSet.describe()

count    928.000000
mean       0.015861
std        0.019352
min        0.000577
25%        0.007638
50%        0.008976
75%        0.011011
max        0.091042
Name: ts_min, dtype: float64

In [58]:
a = thresholdSet.index

focusGroup_2 = time_table_1[time_table_1["to_account"].isin(a)]
focusGroup_2


,to_account,timestamp,ts_min,gap
1455667,8000FE640,2022-09-01 22:34:00,27701.194,NaN
1455664,8000FE640,2022-09-01 22:37:00,27701.197,3.0
1455663,8000FE640,2022-09-01 22:38:00,27701.198,1.0
1455665,8000FE640,2022-09-01 22:51:00,27701.211,13.0
1455666,8000FE640,2022-09-01 22:55:00,27701.215,4.0
...,...,...,...,...
4114760,81C135521,2022-09-06 11:45:00,27707.745,13.0
4114759,81C135521,2022-09-06 11:48:00,27707.748,3.0
1691194,81C194871,2022-09-02 03:08:00,27701.468,NaN
1691193,81C194871,2022-09-02 03:12:00,27701.472,4.0


In [59]:
focusGroup_2["gap"].describe()

count    9217.000000
mean        4.877509
std        25.925827
min         0.000000
25%         0.000000
50%         1.000000
75%         3.000000
max       706.000000
Name: gap, dtype: float64

In [60]:
focusGroup_2["gap"].quantile([0.99,0.98,0.95,0.9,0.8,0.7])

0.99    114.84
0.98     21.00
0.95     11.00
0.90      6.00
0.80      4.00
0.70      2.00
Name: gap, dtype: float64

Clustering

In [61]:
!pip install scikit-learn



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [62]:
from sklearn.cluster import DBSCAN

I am going to try first on a single row to test it

In [63]:
singleAccount = focusGroup_2["to_account"].iloc[0] #this is taking the first value of to_account

data_1 = focusGroup_2[focusGroup_2["to_account"] == singleAccount]

data_1

,to_account,timestamp,ts_min,gap
1455667,8000FE640,2022-09-01 22:34:00,27701.194,NaN
1455664,8000FE640,2022-09-01 22:37:00,27701.197,3.0
1455663,8000FE640,2022-09-01 22:38:00,27701.198,1.0
1455665,8000FE640,2022-09-01 22:51:00,27701.211,13.0
1455666,8000FE640,2022-09-01 22:55:00,27701.215,4.0


In [64]:
reshaped = data_1["ts_min"].values.reshape(-1, 1)
reshaped

array([[27701.194],
       [27701.197],
       [27701.198],
       [27701.211],
       [27701.215]])

In [65]:
model = DBSCAN(eps=6, min_samples=3)
beta_1 = model.fit_predict(reshaped)

print(beta_1)

[0 0 0 0 0]


Creating a function

In [66]:
def cluster(group):
    X = group["ts_min"].values.reshape(-1, 1)
    return pd.Series(
        DBSCAN(eps=2, min_samples=3).fit_predict(X),
        index=group.index
    )

result = time_table_1.groupby("to_account", group_keys=False).apply(cluster)

In [67]:
result.value_counts()

 0    2925316
 1    1840556
 2     384556
-1     134694
 3      32117
 4        885
Name: count, dtype: int64

In [68]:
time_table_1["cluster"] = result

time_table_1

,to_account,timestamp,ts_min,gap,cluster
157311,10042B660,2022-09-01 00:05:00,27699.845,NaN,0
7112,10042B660,2022-09-01 00:08:00,27699.848,3.0,0
72921,10042B660,2022-09-01 00:13:00,27699.853,5.0,0
164374,10042B660,2022-09-01 00:13:00,27699.853,0.0,0
151798,10042B660,2022-09-01 00:18:00,27699.858,5.0,0
...,...,...,...,...,...
5389191,81C1EC560,2022-09-08 10:27:00,27710.547,9.0,1
6038717,81C1EC560,2022-09-09 07:02:00,27711.782,1235.0,1
6038716,81C1EC560,2022-09-09 07:13:00,27711.793,11.0,1
6619796,81C1EC560,2022-09-09 23:01:00,27712.741,948.0,1


In [69]:
time_table_1["from_account"] = focusGroup["account"]
time_table_1

,to_account,timestamp,ts_min,gap,cluster,from_account
157311,10042B660,2022-09-01 00:05:00,27699.845,NaN,0,81A731C90
7112,10042B660,2022-09-01 00:08:00,27699.848,3.0,0,800ABAD90
72921,10042B660,2022-09-01 00:13:00,27699.853,5.0,0,80B676EE0
164374,10042B660,2022-09-01 00:13:00,27699.853,0.0,0,81BD7B530
151798,10042B660,2022-09-01 00:18:00,27699.858,5.0,0,819A0A4F0
...,...,...,...,...,...,...
5389191,81C1EC560,2022-09-08 10:27:00,27710.547,9.0,1,81C1EC560
6038717,81C1EC560,2022-09-09 07:02:00,27711.782,1235.0,1,81C1EC560
6038716,81C1EC560,2022-09-09 07:13:00,27711.793,11.0,1,81C1EC560
6619796,81C1EC560,2022-09-09 23:01:00,27712.741,948.0,1,81C1EC560


Now I am going to create a new table called summary in which I will group by to_account and cluster and see how many senders there are per cluster, what is its duration and the amount of transactions per cluster.
Thus, I will be able to see if I got day, minutes or hours lasting clusters.

In [184]:
cluster_summary = (time_table_1[time_table_1["cluster"] != -1]
           .groupby(["to_account", "cluster"])
           .agg(senders=("from_account", "nunique"),
                duration_min=("ts_min", lambda s: s.max() - s.min()),
                amount_of_transactions_per_cluster = ("from_account", "count")))

filtered_cluster_summary =  cluster_summary[cluster_summary["senders"] > 2]

In [230]:
cluster_summary.reset_index()["to_account"].nunique()

224698

In [231]:
filtered_cluster_summary

senders  duration_min  amount_of_transactions_per_cluster
to_account cluster                                                           
10042B660  0            785         2.874                                 786
           1            767         1.437                                 767
10042B6A8  0            478         2.537                                 478
           1            473         1.437                                 473
10042B6F0  0            176         2.843                                 176
...                     ...           ...                                 ...
81C1EA950  0              3         4.933                                  18
81C1EAD61  0              3         6.134                                  18
81C1EB101  0              3         4.443                                  12
81C1EB521  0              3         1.453                                   3
81C1EC560  1              3         7.038                                  19

[241152 rows x 3 columns]

In [232]:
filtered_cluster_summary.describe()

,senders,duration_min,amount_of_transactions_per_cluster
count,241152.000000,241152.000000,241152.000000
mean,3.753541,4.743153,13.976587
std,4.142620,4.055491,11.049577
min,3.000000,0.000000,3.000000
25%,3.000000,2.008000,7.000000
50%,3.000000,2.789000,10.000000
75%,4.000000,6.495000,17.000000
max,785.000000,15.417000,786.000000


In [233]:
filtered_cluster_summary["amount_of_transactions_per_cluster"].sum()

np.int64(3370482)

In [234]:
filtered_cluster_summary.reset_index()["to_account"].nunique()

211099